# Lab 10: Predict Days to Pay at Summit Gear

You are the analyst at **Summit Gear Co.** The collections team hands you one table of
2,200 B2B customers and one question: how long will each customer take to pay
(`days_to_pay`)? They will use the expected days to decide who gets a call before the
invoice is late. You specify the model, the AI fits it, and you decide what the fit means
and whether to hand it over.

> **First:** in Colab, choose **File > Save a copy in Drive** and work in your copy. The
> copy that opens from the link does not save your work.

> **Then:** **Runtime > Run all.** The notebook runs clean before you have written
> anything. The **Final check** at the bottom lists what is still yours to do.

The [Lab 10 page](https://seanmccaman.com/acctg6155/2026-fall/week-10/lab/lab_10.html) has the AI tutor, the data dictionary, and what to submit.

## How this lab works

1. **You specify.** Target, predictors, units, and why each one belongs.
2. **The AI writes the fit.** Ask Gemini in Colab, or any AI tool, for the `statsmodels` code, and paste it into the cell marked TODO.
3. **You validate it ran.** The right rows and the right predictors. The cell under each fit prints both.
4. **You interpret.** What the numbers mean for collections, in your own words.

Keep the names the TODO cells give you (`ols_full`, `corr`, `PRUNED_PREDICTORS`,
`ols_pruned`, `ols_train`, `test_pred`) and both `def` lines exactly as given. The Final
check reads them by name.

## Section 0: Setup and load

Colab already has `pandas`, `statsmodels` and `scikit-learn`. On your own machine, run
`pip install pandas statsmodels scikit-learn` once first.

In [ ]:
# Section 0: run once.
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

print("pandas", pd.__version__)
print("setup ok")

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-10/summit_gear_customers.csv"

df = pd.read_csv(DATA_URL)
print("customers:", df.shape)
print("missing values:", int(df.isna().sum().sum()), "  duplicate customer_id:", int(df["customer_id"].duplicated().sum()))
print()
print(df.dtypes)

The target is `days_to_pay`. These are the ten candidate predictors. `defaulted` is
Week 11's target, so leave it out. `tenure_months` and the three category columns stay
out too, so everyone's first fit is the same model. The
[data dictionary](https://github.com/sean-mccaman/acctg5150-090/blob/main/2026-fall/week-10/summit_gear_customers_dictionary.md) has every column and its unit.

In [ ]:
PREDICTORS = [
    "credit_limit", "avg_balance", "annual_spend", "prior_late_count",
    "days_since_last_order", "orders_per_year", "discount_rate",
    "avg_order_value", "noise_a", "noise_b",
]
print(len(PREDICTORS), "candidate predictors")
df[["days_to_pay"] + PREDICTORS].describe().round(2).T

# Exercise 1: Specify and fit (34 points)

The first fit uses all ten candidates on purpose. Exercise 2 cleans the list.

**Specify.** Fill in the model definition below, one line each.

**Model definition (Exercise 1)**

- **Target:** _the column, and its unit_
- **Predictors and units:** _each predictor and its unit_
- **Why each belongs:** _one reason per predictor, or per group of predictors_
- **Look-ahead check:** _would you have each value when you make the prediction?_
- **Benchmark to beat:** _what you would predict with no model at all_

**Fit.** Ask the AI for the `statsmodels` OLS of `days_to_pay` on `PREDICTORS`, with a
constant. Paste it below and name the fitted model `ols_full`.

In [ ]:
# TODO (Exercise 1): paste the AI's fit here.
# A prompt that works: "Write the statsmodels OLS that regresses days_to_pay on the
# columns in PREDICTORS, with a constant, and print the summary."
ols_full = None   # replace None with the fitted model

In [ ]:
# Validate: the right rows and the right predictors?
if ols_full is not None and hasattr(ols_full, "nobs"):
    print("rows used:", int(ols_full.nobs), "(should be 2200)")
    print("predictors:", [p for p in ols_full.params.index if p not in ("const", "Intercept")])
    print("R-squared:", round(ols_full.rsquared, 4), "  adjusted R-squared:", round(ols_full.rsquared_adj, 4))
else:
    print("ols_full is not a statsmodels fit yet. Ask for the statsmodels OLS and paste it above.")

**Interpret.** Read two coefficients in collections terms: days per unit, holding the
other predictors fixed. `prior_late_count` and `discount_rate` read naturally. If you pick
a dollar column, read it per $1,000. Then say what R-squared tells collections, in one
sentence.

**Your interpretation (Exercise 1)**

_Replace this line with your answer._

# Exercise 2: Clean the predictor set (32 points)

Some of the candidates measure the same thing. When predictors move together, the model
cannot separate their effects, and their coefficients get unstable. A correlation matrix
finds them.

Ask the AI for the correlation matrix of `PREDICTORS`. Name it `corr`.

In [ ]:
# TODO (Exercise 2): paste the AI's correlation matrix here.
# A prompt that works: "Give me the correlation matrix of the columns in PREDICTORS
# as a DataFrame, and print it rounded to two decimals."
corr = None   # replace None with the correlation matrix

## Helper: `flag_collinear_pairs`

Write this one yourself. It reads a correlation matrix and lists the pairs that move together.

- **Takes:** `corr`, a correlation DataFrame like `df[PREDICTORS].corr()`, and `threshold`, default `0.9`.
- **Returns:** a list of `(col_a, col_b, r)` tuples, one for each pair above the diagonal whose `abs(r)` is at least `threshold`.
- **Example:** at `0.9`, a pair at `0.98` is in, a pair at `-0.95` is in, and a pair at `0.20` is out.

Read only above the diagonal (`j > i`), so each pair counts once and the `1.0` on the
diagonal is skipped. Grading runs it on correlation matrices you have not seen.

In [ ]:
def flag_collinear_pairs(corr, threshold=0.9):
    """Flag predictor pairs that move together, from a correlation matrix.

    Takes:
      corr      -- a pandas correlation DataFrame (e.g. df[predictors].corr()),
                   square, with matching row and column labels.
      threshold -- float, default 0.9.
    Returns:
      list of (col_a, col_b, r) tuples -- one per UNIQUE upper-triangle pair whose
      absolute correlation abs(r) is at least threshold. col_a and col_b are the two
      predictor names (strings); r is their correlation (float).
    """
    return None

In [ ]:
# Your helper on the real matrix.
if corr is not None and flag_collinear_pairs(corr) is not None:
    for a, b, r in flag_collinear_pairs(corr, threshold=0.9):
        print(f"{a} & {b}: {r:.3f}")
else:
    print("Build corr above and write flag_collinear_pairs, then run this again.")

**Decide and re-fit.** The pruned model is **one** of the account-size columns your helper
flags, plus the four payment-behavior columns: `prior_late_count`,
`days_since_last_order`, `orders_per_year`, `discount_rate`. Which size column to keep is
your call. Drop `avg_order_value` and the two noise columns as well. Look at their
p-values in the full fit: none of the three is distinguishable from zero. Put your list in
`PRUNED_PREDICTORS`, then have the AI re-fit the OLS on it as `ols_pruned`.

In [ ]:
# TODO (Exercise 2): your pruned list, then the AI's re-fit.
PRUNED_PREDICTORS = None   # one size column plus the four payment-behavior columns
ols_pruned = None          # replace None with the OLS of days_to_pay on PRUNED_PREDICTORS

In [ ]:
# Compare the two fits.
if hasattr(ols_full, "rsquared_adj") and hasattr(ols_pruned, "rsquared_adj"):
    print("adjusted R-squared, full  :", round(ols_full.rsquared_adj, 4))
    print("adjusted R-squared, pruned:", round(ols_pruned.rsquared_adj, 4))
    print("pruned predictors:", [p for p in ols_pruned.params.index if p not in ("const", "Intercept")])
else:
    print("Fit ols_full and ols_pruned first, both as statsmodels fits.")

**Justify.** One paragraph: which columns the matrix flagged and the r values that told
you, which one you kept and why, and what adjusted R-squared did when you pruned. If it
fell, say what that tells you about the column you kept.

**Your justification (Exercise 2)**

_Replace this line with your answer._

# Exercise 3: Judge it on data the model has not seen (32 points)

R-squared is measured on rows the model already saw. Collections will use it on
customers it has not seen. So hold some rows out: fit on 80 percent, then score the other
20 percent against the benchmark, the training mean.

The split is fixed so everyone scores the same rows. Run it as given.

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=6155)
print("train rows:", len(train), "  test rows:", len(test))

Ask the AI to fit your pruned model on `train` only, as `ols_train`, and to predict the
`test` rows, as `test_pred`.

In [ ]:
# TODO (Exercise 3): paste the AI's code here.
# A prompt that works: "Fit the statsmodels OLS of days_to_pay on PRUNED_PREDICTORS using
# only the train DataFrame, with a constant, then predict days_to_pay for the test DataFrame."
ols_train = None   # the pruned model, fit on train only
test_pred = None   # its predictions for the test rows

## Helper: `rmse`

Root mean squared error is the typical miss, in days. Write it yourself.

- **Takes:** `actual` and `predicted`, two sequences of numbers the same length (lists, numpy arrays, or pandas Series).
- **Returns:** a float, the square root of the mean of `(actual - predicted) ** 2`.
- **Example:** `rmse([10, 20], [12, 18])` is `2.0`.

Grading runs it on numbers you have not seen.

In [ ]:
def rmse(actual, predicted):
    """Root mean squared error between actual and predicted values.

    Takes:
      actual    -- a sequence of numbers (list, numpy array, or pandas Series).
      predicted -- a sequence of numbers the same length as actual.
    Returns:
      float -- the square root of the mean of (actual - predicted) ** 2.
    """
    return None

In [ ]:
# Score the model and the benchmark on the same test rows.
test_rmse_model, test_rmse_naive = None, None
if test_pred is not None:
    naive_pred = np.full(len(test), train["days_to_pay"].mean())   # the benchmark: the training mean
    test_rmse_model = rmse(test["days_to_pay"], test_pred)
    test_rmse_naive = rmse(test["days_to_pay"], naive_pred)
print("test RMSE, your model:", test_rmse_model)
print("test RMSE, benchmark :", test_rmse_naive)

**Decide.** One paragraph to the collections manager:

- Would you hand this model over? Compare its test RMSE with the benchmark's.
- Name one thing that moves days to pay that a customer attribute table cannot know.
- If you adjusted the model's number for that, how would you document it: the assumption, the amount, and the source?

**Your decision (Exercise 3)**

_Replace this line with your answer._

# Final check

Run the cell below. You do not edit it. It prints `RESULTS`, the numbers grading reads,
and runs your two helpers on one fixed input each. Use the **Table of Contents** (the list
icon at the top left in Colab) to jump back to an exercise.

In [ ]:
# ===== Final check. Run it. You do not edit this cell. =====
import json
from IPython.display import display, HTML

SIZE_COLUMNS = ["credit_limit", "avg_balance", "annual_spend"]


def final_check():
    """Collect RESULTS, run both helpers on one fixed input each, and show what is left."""
    g = globals()

    def num(x, places=4):
        try:
            return round(float(x), places)
        except Exception:
            return None

    def names(model):
        try:
            return [p for p in model.params.index if p not in ("const", "Intercept")]
        except Exception:
            return None

    full, pruned = g.get("ols_full"), g.get("ols_pruned")
    train_df, test_df = g.get("train"), g.get("test")
    full_names, pruned_names = names(full), names(pruned)
    kept = None
    if pruned_names is not None:
        hits = [c for c in SIZE_COLUMNS if c in pruned_names]
        kept = hits[0] if len(hits) == 1 else (hits or None)

    results = {
        "r2_full": num(getattr(full, "rsquared", None)),
        "adj_r2_full": num(getattr(full, "rsquared_adj", None)),
        "adj_r2_pruned": num(getattr(pruned, "rsquared_adj", None)),
        "test_rmse_model": num(g.get("test_rmse_model")),
        "test_rmse_naive": num(g.get("test_rmse_naive")),
        "n_train": None if train_df is None else int(len(train_df)),
        "n_test": None if test_df is None else int(len(test_df)),
        "kept_size_predictor": kept,
    }

    # Both helpers, on one fixed input each.
    labels = ["a", "b", "c", "d"]
    fixed_corr = pd.DataFrame([[1.00, 0.95, 0.10, 0.20],
                               [0.95, 1.00, 0.30, -0.93],
                               [0.10, 0.30, 1.00, 0.40],
                               [0.20, -0.93, 0.40, 1.00]], index=labels, columns=labels)
    try:
        got = flag_collinear_pairs(fixed_corr, 0.9)
        flag_out = None if got is None else [[str(a), str(b), num(r)] for a, b, r in got]
    except Exception as e:
        flag_out = "error: " + type(e).__name__
    try:
        rmse_out = num(rmse([30, 40, 50, 60], [33, 36, 50, 61]))
    except Exception as e:
        rmse_out = "error: " + type(e).__name__
    helpers = {"flag_collinear_pairs": flag_out, "rmse": rmse_out}

    def ok_full():
        return int(full.nobs) == 2200 and sorted(full_names) == sorted(PREDICTORS)

    def ok_corr():
        c = g.get("corr")
        return c is not None and tuple(c.shape) == (10, 10)

    def ok_flag():
        return (isinstance(flag_out, list) and len(flag_out) == 2
                and {(a, b) for a, b, _ in flag_out} == {("a", "b"), ("b", "d")})

    def ok_pruned():
        return (int(pruned.nobs) == 2200 and isinstance(kept, str)
                and not {"noise_a", "noise_b"} & set(pruned_names))

    def ok_split():
        return results["n_train"] == 1760 and results["n_test"] == 440

    def ok_rmse():
        return isinstance(rmse_out, float) and abs(rmse_out - 2.5495) < 0.001

    def ok_scores():
        m, b = results["test_rmse_model"], results["test_rmse_naive"]
        return m is not None and b is not None and m < b

    checks = [
        ("ols_full: 2,200 rows, the ten PREDICTORS", "Exercise 1", ok_full),
        ("corr: a 10 by 10 correlation matrix",      "Exercise 2", ok_corr),
        ("flag_collinear_pairs: fixed example",      "Exercise 2", ok_flag),
        ("ols_pruned: one size column, no noise",    "Exercise 2", ok_pruned),
        ("train and test: 1,760 and 440 rows",       "Exercise 3", ok_split),
        ("rmse: fixed example",                      "Exercise 3", ok_rmse),
        ("test RMSE: model and benchmark scored",    "Exercise 3", ok_scores),
    ]
    rows, passed, todo = [], 0, []
    for label, where, test in checks:
        try:
            good = bool(test())
        except Exception:
            good = False
        if good:
            passed += 1
            rows.append("<tr><td style='font-size:18px;color:#176b3a'>&#10003;</td>"
                        "<td style='padding:2px 14px'>{}</td>"
                        "<td style='color:#176b3a'>done</td></tr>".format(label))
        else:
            todo.append("{} ({})".format(label, where))
            rows.append("<tr><td style='font-size:18px;color:#BE0000'>&#10007;</td>"
                        "<td style='padding:2px 14px'>{}</td>"
                        "<td style='color:#BE0000'>not yet, see <b>{}</b></td></tr>".format(label, where))
    color = "#176b3a" if passed == len(checks) else "#BE0000"
    display(HTML(
        "<div style=\"font-family:Arial,Helvetica,sans-serif\">"
        "<h3 style='margin:4px 0;color:{}'>Lab 10 check: {} of {} done</h3>"
        "<table style='border-collapse:collapse;font-size:14px'>{}</table>"
        "<p style='font-size:12.5px;color:#555;margin-top:8px'>A green row means that piece ran "
        "and has the right shape. It does not grade your numbers or your writing. Grading checks "
        "RESULTS against the key for the size column you kept, runs both helpers on inputs you have "
        "not seen, and reads your four written cells.</p></div>".format(color, passed, len(checks), "".join(rows))))
    print("RESULTS =", json.dumps(results))
    print("HELPERS =", json.dumps(helpers))
    print("PRUNED_PREDICTORS =", json.dumps(pruned_names))
    if todo:
        print("Still to do:", "; ".join(todo))
    return results


RESULTS = final_check()

## Before you submit

1. **Runtime > Restart session and run all**, so every cell shows its output and the Final check reads your latest work.
2. The Final check prints `RESULTS` with no `null` in it, and every row is green.
3. Your four written cells have your words in them, not the placeholder line.
4. **File > Download > Download .ipynb.** Rename it `<uID>_lab_10.ipynb`, with your uID in place of `<uID>`, and upload it to Lab 10 in Canvas.